In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from clickhouse_connect import get_client
import seaborn as sns
import ipaddress
import json
import textwrap

In [2]:
def count_valid_records(json_results_str):
    try:
        records = json.loads(json_results_str)
        if not isinstance(records, list):
            return 0
        valid_count = 0
        for record in records:
            if isinstance(record, dict) and 'error' not in record:
                valid_count += 1
        return valid_count
    except (json.JSONDecodeError, TypeError):
        return 0

def check_for_ech_key(json_results_str):
    try:
        records = json.loads(json_results_str)
        if not isinstance(records, list):
            return False
        for record in records:
            if isinstance(record, dict) and 'ECH' in record:
                return True
        return False
    except (json.JSONDecodeError, TypeError):
        return False

def extract_ips_from_dns_results(json_results_str, ip_version_type):
    ips = []
    try:
        records = json.loads(json_results_str)
        if not isinstance(records, list):
            return ips
        for record in records:
            if isinstance(record, dict) and 'error' not in record and 'ip' in record:
                try:
                    ip_obj = ipaddress.ip_address(record['ip'])
                    if (ip_version_type == 'IPv4' and isinstance(ip_obj, ipaddress.IPv4Address)) or \
                       (ip_version_type == 'IPv6' and isinstance(ip_obj, ipaddress.IPv6Address)):
                        ips.append(record['ip'])
                except ValueError:
                    pass
    except (json.JSONDecodeError, TypeError):
        pass
    return ips

In [3]:
client = get_client(host='localhost', port=8123, username='default', password='default')

query = """
        SELECT
            worker_id,
            run_uuid,
            domain,
        start,
        end,
        dns_a_results,
        dns_aaaa_results,
        dns_svcb_results,
        dns_https_results
    FROM dns_results \
        """

df = client.query_df(query)
df.head(10)

,worker_id,run_uuid,domain,start,end,dns_a_results,dns_aaaa_results,dns_svcb_results,dns_https_results
0,node-1,e03e3152-af55-4a84-af30-6c5aa5fef58b,ielts.com,2025-07-14 10:12:53,2025-07-14 10:12:53,"[{""ip"": ""107.21.158.65""}, {""ip"": ""18.204.86.12...",[],[],[]
1,node-1,e03e3152-af55-4a84-af30-6c5aa5fef58b,ejendals.com,2025-07-14 10:12:53,2025-07-14 10:12:53,"[{""ip"": ""20.240.181.243""}]",[],[],[]
2,node-1,e03e3152-af55-4a84-af30-6c5aa5fef58b,our-vps.ru,2025-07-14 10:12:53,2025-07-14 10:12:53,"[{""ip"": ""37.1.218.29""}]",[],[],[]
3,node-1,e03e3152-af55-4a84-af30-6c5aa5fef58b,w88you.com,2025-07-14 10:12:53,2025-07-14 10:12:53,"[{""ip"": ""146.190.201.74""}]",[],[],[]
4,node-1,e03e3152-af55-4a84-af30-6c5aa5fef58b,windmillair.com,2025-07-14 10:12:53,2025-07-14 10:12:53,"[{""ip"": ""23.227.38.32""}]",[],[],[]
5,node-1,e03e3152-af55-4a84-af30-6c5aa5fef58b,imarketing.courses,2025-07-14 10:12:53,2025-07-14 10:12:53,"[{""ip"": ""172.66.41.7""}, {""ip"": ""172.66.42.249""}]","[{""ip"": ""2606:4700:3108::ac42:2907""}, {""ip"": ""...",[],"[{""ALPN"": ""h3,h2"", ""IPV4HINT"": ""172.66.41.7,17..."
6,node-1,e03e3152-af55-4a84-af30-6c5aa5fef58b,motionview.com.bd,2025-07-14 10:12:53,2025-07-14 10:12:53,"[{""ip"": ""104.21.11.132""}, {""ip"": ""172.67.149.6...","[{""ip"": ""2606:4700:3036::6815:b84""}, {""ip"": ""2...",[],"[{""ALPN"": ""h3,h2"", ""IPV4HINT"": ""104.21.11.132,..."
7,node-1,e03e3152-af55-4a84-af30-6c5aa5fef58b,rockher.com,2025-07-14 10:12:53,2025-07-14 10:12:53,"[{""ip"": ""23.227.38.65""}]",[],[],[]
8,node-1,e03e3152-af55-4a84-af30-6c5aa5fef58b,prava112ab.com,2025-07-14 10:12:53,2025-07-14 10:12:53,"[{""ip"": ""185.149.120.117""}]",[],[],[]
9,node-1,e03e3152-af55-4a84-af30-6c5aa5fef58b,spotinc.com,2025-07-14 10:12:53,2025-07-14 10:12:53,"[{""ip"": ""35.237.26.109""}]",[],[],[]


In [4]:
has_a_error    = df['dns_a_results'].str.contains('"error"', na=False)
has_aaaa_error = df['dns_aaaa_results'].str.contains('"error"', na=False)
has_svcb_error = df['dns_svcb_results'].str.contains('"error"', na=False)
has_https_error= df['dns_https_results'].str.contains('"error"', na=False)

df['has_error'] = has_a_error | has_aaaa_error | has_svcb_error | has_https_error

df_successful   = df[~df['has_error']].copy()
df_unsuccessful = df[df['has_error']].copy()

df_successful['svcb_answer_count']          = df_successful['dns_svcb_results'].apply(count_valid_records)
df_successful['https_answer_count_general'] = df_successful['dns_https_results'].apply(count_valid_records)
df_successful['ipv4_answer_count']          = df_successful['dns_a_results'].apply(count_valid_records)
df_successful['ipv6_answer_count']          = df_successful['dns_aaaa_results'].apply(count_valid_records)

df_successful['has_svcb_entry']  = df_successful['svcb_answer_count'] > 0
df_successful['has_https_entry'] = df_successful['https_answer_count_general'] > 0
df_successful['has_ipv4']        = df_successful['ipv4_answer_count'] > 0
df_successful['has_ipv6']        = df_successful['ipv6_answer_count'] > 0

df_successful['has_ech_in_https'] = df_successful['dns_https_results'].apply(check_for_ech_key)
df_successful['has_ech_in_svcb']  = df_successful['dns_svcb_results'].apply(check_for_ech_key)
df_successful['has_ech']          = df_successful['has_ech_in_svcb'] | df_successful['has_ech_in_https']

### DNS resolution success rate and presence of record types among responsive domains

In [5]:
sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(1, 2, figsize=(9, 4), gridspec_kw={'width_ratios': [1, 3]})

success_df_plot = df['has_error'].value_counts(normalize=True).reset_index()
success_df_plot.columns = ['is_error','Percentage']
success_df_plot['Status'] = success_df_plot['is_error'].map({
    False: 'Successful', True: 'Unsuccessful'
})
success_df_plot = success_df_plot.sort_values(by='is_error', ascending=True)

colors = sns.color_palette("Greys", n_colors=len(success_df_plot))

sns.barplot(
    x='Status',
    y='Percentage',
    data=success_df_plot,
    palette=colors,
    ax=axes[0]
)
axes[0].set_ylabel('Percentage (%)', fontsize=12)
axes[0].set_xlabel('', fontsize=12)
axes[0].set_ylim(0, 1)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.1%}') )

for idx, p in enumerate(axes[0].patches):
    axes[0].text(
        p.get_x() + p.get_width() / 2.,
        p.get_height(),
        f'{p.get_height():.1%}',
        fontsize=12,
        color='black',
        ha='center',
        va='bottom'
    )

total = len(df_successful)

metrics = {
    'SVCB Records': ('has_svcb_entry',  df_successful['has_svcb_entry'].sum()),
    'HTTPS Records':('has_https_entry', df_successful['has_https_entry'].sum()),
    'IPv4 Records': ('has_ipv4',        df_successful['has_ipv4'].sum()),
    'IPv6 Records': ('has_ipv6',        df_successful['has_ipv6'].sum()),
}

presence_data = pd.DataFrame([
    {'Record Type': rt, 'Domains': count, 'Percentage (%)': count/total*100}
    for rt, (col, count) in metrics.items()
])

sns.barplot(
    x='Record Type',
    y='Percentage (%)',
    data=presence_data,
    palette=sns.color_palette("Greys", len(presence_data)),
    ax=axes[1]
)
axes[1].set_ylim(0, 100)
axes[1].set_ylabel('Percentage (%)', fontsize=12)
axes[1].set_xlabel('Record Type', fontsize=12)

for idx, row in presence_data.iterrows():
    axes[1].text(
        idx, row['Percentage (%)'] + 1,
        f"{row['Percentage (%)']:.2f}%",
        ha='center', va='bottom', fontsize=12
    )

plt.tight_layout()
plt.show()

print("\n--- Overall Query Success/Failure ---")
success_df_table = df['has_error'].value_counts(normalize=False).reset_index()
success_df_table.columns = ['Status_Bool','Count']
success_df_table['Status']      = success_df_table['Status_Bool'].map({
    False: 'Successful', True: 'Unsuccessful'
})
success_df_table['Percentage']  = (df['has_error'].value_counts(normalize=True)*100).values
success_df_table = success_df_table[['Status','Count','Percentage']]
print(success_df_table.to_string(index=False))

print("\n--- Presence of Record Types in Successful Queries ---")
presence_summary = pd.DataFrame([
    {'Metric':        'Total Successful Domains',    'Value': total},
    {'Metric':        'Domains with SVCB Entry',     'Value': metrics['SVCB Records'][1]},
    {'Metric':        'Percent with SVCB Entry',     'Value': f"{presence_data.loc[presence_data['Record Type']=='SVCB Records','Percentage (%)'].iloc[0]:.2f}%"},
    {'Metric':        'Domains with HTTPS Entry',    'Value': metrics['HTTPS Records'][1]},
    {'Metric':        'Percent with HTTPS Entry',    'Value': f"{presence_data.loc[presence_data['Record Type']=='HTTPS Records','Percentage (%)'].iloc[0]:.2f}%"},
    {'Metric':        'Domains with IPv4 Records',   'Value': metrics['IPv4 Records'][1]},
    {'Metric':        'Percent with IPv4 Records',   'Value': f"{presence_data.loc[presence_data['Record Type']=='IPv4 Records','Percentage (%)'].iloc[0]:.2f}%"},
    {'Metric':        'Domains with IPv6 Records',   'Value': metrics['IPv6 Records'][1]},
    {'Metric':        'Percent with IPv6 Records',   'Value': f"{presence_data.loc[presence_data['Record Type']=='IPv6 Records','Percentage (%)'].iloc[0]:.2f}%"},
])
print(presence_summary.to_string(index=False))



--- Overall Query Success/Failure ---
      Status  Count  Percentage
  Successful 995255     99.5255
Unsuccessful   4745      0.4745

--- Presence of Record Types in Successful Queries ---
                   Metric  Value
 Total Successful Domains 995255
  Domains with SVCB Entry     59
  Percent with SVCB Entry  0.01%
 Domains with HTTPS Entry 229751
 Percent with HTTPS Entry 23.08%
Domains with IPv4 Records 883366
Percent with IPv4 Records 88.76%
Domains with IPv6 Records 301230
Percent with IPv6 Records 30.27%


### Fraction of domains with SVCB or HTTPS record types that include ECH configuration, and overall ECH support among responsive domains.

In [6]:
total_successful_domains = len(df_successful)

svc_domains_df = df_successful[df_successful['has_svcb_entry']]
svc_with_ech_count = svc_domains_df['has_ech_in_svcb'].sum()
svc_total_domains = len(svc_domains_df)
svc_without_ech_count = svc_total_domains - svc_with_ech_count

https_domains_df = df_successful[df_successful['has_https_entry']]
https_with_ech_count = https_domains_df['has_ech_in_https'].sum()
https_total_domains = len(https_domains_df)
https_without_ech_count = https_total_domains - https_with_ech_count

all_successful_with_ech = df_successful['has_ech'].sum()
all_successful_without_ech = total_successful_domains - all_successful_with_ech

combined_plot_data = pd.DataFrame({
    'Category': [
        'Domains with SVCB Records',
        'Domains with HTTPS Records',
        'All Successful Domains'
    ],
    'With ECH': [
        svc_with_ech_count,
        https_with_ech_count,
        all_successful_with_ech
    ],
    'Without ECH': [
        svc_without_ech_count,
        https_without_ech_count,
        all_successful_without_ech
    ],
    'Total': [
        svc_total_domains,
        https_total_domains,
        total_successful_domains
    ]
})

combined_plot_data['With ECH %']    = (combined_plot_data['With ECH'] / combined_plot_data['Total']) * 100
combined_plot_data['Without ECH %'] = (combined_plot_data['Without ECH'] / combined_plot_data['Total']) * 100
combined_plot_data.loc[combined_plot_data['Total'] == 0, ['With ECH %', 'Without ECH %']] = 0

sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(40, 10))

colors = sns.color_palette("Greys", 2)
with_ech_color, without_ech_color = colors[1], colors[0]

wrapped_categories = [textwrap.fill(label, 20) for label in combined_plot_data['Category']]

ax.barh(
    wrapped_categories,
    combined_plot_data['With ECH %'],
    label='With ECH Key',
    color=with_ech_color,
    edgecolor='black', linewidth=0.5, height=0.5
)

ax.barh(
    wrapped_categories,
    combined_plot_data['Without ECH %'],
    left=combined_plot_data['With ECH %'],
    label='Without ECH Key',
    color=without_ech_color,
    edgecolor='black', linewidth=0.5, height=0.5
)

ax.set_xlabel('Percentage (%)', fontsize=20)
ax.set_ylabel('')
ax.set_xlim(0, 100)
ax.tick_params(axis='y', labelsize=20)
ax.tick_params(axis='x', labelsize=20)

ax.invert_yaxis()

for i, row in combined_plot_data.iterrows():
    if row['With ECH %'] > 0:
        ax.text(
            row['With ECH %'] / 2,
            i,
            f"{row['With ECH %']:.1f}%",
            ha='center', va='center',
            color='white', fontsize=20, fontweight='bold'
        )
    if row['Without ECH %'] > 0:
        if row['Without ECH %'] >= 5:
            ax.text(
                row['With ECH %'] + row['Without ECH %'] / 2,
                i,
                f"{row['Without ECH %']:.1f}%",
                ha='center', va='center',
                color='black', fontsize=20, fontweight='bold'
            )

ax.legend(
    title='ECH Status',
    loc='lower center',
    bbox_to_anchor=(0.5, -0.3),
    ncol=2,
    fontsize=20,
    title_fontsize=22
)

plt.tight_layout(rect=[0, 0.7, 0.5, 0.5])
plt.show()

print("\n--- Overall Query Success/Failure ---")
success_df_table = df['has_error'].value_counts(normalize=False).reset_index()
success_df_table.columns = ['Status_Bool','Count']
success_df_table['Status']      = success_df_table['Status_Bool'].map({
    False: 'Successful', True: 'Unsuccessful'
})
success_df_table['Percentage']  = (df['has_error'].value_counts(normalize=True)*100).values
success_df_table = success_df_table[['Status','Count','Percentage']]
print(success_df_table.to_string(index=False))

print("\n--- ECH Presence Summary ---")

ech_summary_data = {
    'Metric': [
        'Total Successful Domains',
        'Domains with SVCB Entry & ECH',
        'Domains with HTTPS Entry & ECH',
        'All Successful Domains with ECH'
    ],
    'Value': [
        total_successful_domains,
        combined_plot_data.loc[combined_plot_data['Category'] == 'Domains with SVCB Records', 'With ECH'].iloc[0],
        combined_plot_data.loc[combined_plot_data['Category'] == 'Domains with HTTPS Records', 'With ECH'].iloc[0],
        combined_plot_data.loc[combined_plot_data['Category'] == 'All Successful Domains', 'With ECH'].iloc[0]
    ]
}
ech_summary = pd.DataFrame(ech_summary_data)
ech_summary['Value'] = ech_summary['Value'].astype(str)

all_ech_count = int(combined_plot_data.loc[
    combined_plot_data['Category'] == 'All Successful Domains', 'With ECH'
].iloc[0])
all_ech_pct = combined_plot_data.loc[
    combined_plot_data['Category'] == 'All Successful Domains', 'With ECH %'
].iloc[0]

ech_summary.loc[ech_summary['Metric'] == 'All Successful Domains with ECH', 'Value'] = \
    f"{all_ech_count} ({all_ech_pct:.2f}%)"

print(ech_summary.to_string(index=False))


--- Overall Query Success/Failure ---
      Status  Count  Percentage
  Successful 995255     99.5255
Unsuccessful   4745      0.4745

--- ECH Presence Summary ---
                         Metric           Value
       Total Successful Domains          995255
  Domains with SVCB Entry & ECH               1
 Domains with HTTPS Entry & ECH          154981
All Successful Domains with ECH 154981 (15.57%)
